# Phase 6: Feature Engineering

Feature engineering transforms existing variables into useful inputs that can help machine-learning models learn patterns more effectively.

In this phase, date-based features will be extracted from the `date` column. The original cleaned dataset will remain unchanged, while a new feature-engineered dataset will be created for modelling.

## Import Libraries

Pandas is used for loading, inspecting, transforming, and saving the dataset.

`Path` is used to create reliable file paths within the project.

In [36]:
from pathlib import Path

import pandas as pd

## Define Project Paths

The project paths allow the notebook to locate the cleaned dataset and save the feature-engineered dataset in the correct folders.

In [37]:
# Creating the project paths

current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

cleaned_data_path = (
    project_root
    / "data"
    / "processed"
    / "walmart_cleaned.csv"
)

feature_data_path = (
    project_root
    / "data"
    / "processed"
    / "walmart_featured.csv"
)

print("Project root:", project_root)
print("Cleaned data path:", cleaned_data_path)
print("Feature data path:", feature_data_path)

Project root: c:\Users\Udoch\walmart-sales-regression
Cleaned data path: c:\Users\Udoch\walmart-sales-regression\data\processed\walmart_cleaned.csv
Feature data path: c:\Users\Udoch\walmart-sales-regression\data\processed\walmart_featured.csv


## Validate the Input File

Before loading the dataset, the notebook checks that the cleaned CSV file exists.

In [38]:
print("Cleaned data path:", cleaned_data_path.resolve())
print("File exists:", cleaned_data_path.exists())

Cleaned data path: C:\Users\Udoch\walmart-sales-regression\data\processed\walmart_cleaned.csv
File exists: True


In [39]:
# Confirming that the cleaned dataset exists

if not cleaned_data_path.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found at: {cleaned_data_path}"
    )

print("Cleaned dataset found successfully.")

Cleaned dataset found successfully.


## Loading the Cleaned Dataset

The cleaned dataset produced during Phase 4 is loaded as the starting point for feature engineering.

In [40]:
df = pd.read_csv(cleaned_data_path)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully.
Dataset shape: (6435, 8)


,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106
1,2,2010-02-05,2136989.46,0,40.19,2.572,210.752605,8.324
2,3,2010-02-05,461622.22,0,45.71,2.572,214.424881,7.368
3,4,2010-02-05,2135143.87,0,43.76,2.598,126.442065,8.623
4,5,2010-02-05,317173.10,0,39.70,2.572,211.653972,6.566


## Inspect the Columns

The available columns are checked before creating new features.

In [41]:
df.columns.tolist()

['store',
 'date',
 'weekly_sales',
 'holiday_flag',
 'temperature',
 'fuel_price',
 'cpi',
 'unemployment']

## Inspect Data Types

The `date` column may be loaded from the CSV file as text. It must be converted back to a datetime data type before extracting date-based features.

In [42]:
df.dtypes

store             int64
date                str
weekly_sales    float64
holiday_flag      int64
temperature     float64
fuel_price      float64
cpi             float64
unemployment    float64
dtype: object

## Convert the Date Column

The `date` column may appear as Str(Object). So i need to convert it to a proper Pandas datetime type before creating date-based features. so, the `date` column is converted from text into a datetime data type so that year, month, week, and quarter information can be extracted.

In [43]:
df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

print("Date data type:", df["date"].dtype)
print("Invalid dates:", df["date"].isna().sum())

Date data type: datetime64[us]
Invalid dates: 0


## Create a Feature-Engineering Copy

A separate copy is created so that the loaded cleaned dataset remains available for comparison.

In [44]:
# Creating a separate working copy

feature_df = df.copy()

print("Feature-engineering copy created.")

Feature-engineering copy created.


### Creating date-based features

The date itself contains useful information. We will extract:

year
month
week of year
quarter
time trend

## Creating the Year Feature

The year feature allows the model to learn whether sales levels changed between 2010, 2011, and 2012.

In [45]:
# Creating the Year Feature

feature_df["year"] = feature_df["date"].dt.year

feature_df[
    ["date", "year"]
].head()

,date,year
0,2010-02-05,2010
1,2010-02-05,2010
2,2010-02-05,2010
3,2010-02-05,2010
4,2010-02-05,2010


## Create the Month Feature

The month feature helps the model learn seasonal sales patterns occurring during different parts of the year.

In [46]:
#Creating the Month Feature

feature_df["month"] = feature_df["date"].dt.month

feature_df[
    ["date", "month"]
].head()

,date,month
0,2010-02-05,2
1,2010-02-05,2
2,2010-02-05,2
3,2010-02-05,2
4,2010-02-05,2


## Creating the Week-of-Year Feature

The week-of-year feature gives a more detailed representation of seasonal timing than the month feature.

Its values generally range from 1 to 52 or 53.

In [47]:
# Creating the Week-of-Year Feature

feature_df["week_of_year"] = (
    feature_df["date"]
    .dt
    .isocalendar()
    .week
    .astype(int)
)

feature_df[
    ["date", "week_of_year"]
].head()

,date,week_of_year
0,2010-02-05,5
1,2010-02-05,5
2,2010-02-05,5
3,2010-02-05,5
4,2010-02-05,5


## Creating the Quarter Feature

The quarter feature groups months into four broader business periods:

- Quarter 1: January to March
- Quarter 2: April to June
- Quarter 3: July to September
- Quarter 4: October to December

In [48]:
# Creating the Quarter Feature

feature_df["quarter"] = feature_df["date"].dt.quarter

feature_df[
    ["date", "quarter"]
].head()

,date,quarter
0,2010-02-05,1
1,2010-02-05,1
2,2010-02-05,1
3,2010-02-05,1
4,2010-02-05,1


## Checking the Day of the Week

Before keeping a weekday feature, its number of unique values is checked.

A feature containing the same value in every row would provide no useful predictive information.

In [49]:
# Checking the Day of the Week

feature_df["day_of_week"] = (
    feature_df["date"].dt.dayofweek
)

print(
    "Unique day-of-week values:",
    feature_df["day_of_week"].unique()
)

print(
    "Number of unique values:",
    feature_df["day_of_week"].nunique()
)

Unique day-of-week values: [4]
Number of unique values: 1


## Removing the Constant Weekday Feature

The dataset records weekly observations on the same day of the week.

Because `day_of_week` contains only one unique value, it cannot help distinguish one observation from another and is removed.

In [50]:
if feature_df["day_of_week"].nunique() == 1:
    feature_df = feature_df.drop(
        columns="day_of_week"
    )

print(feature_df.columns.tolist())

['store', 'date', 'weekly_sales', 'holiday_flag', 'temperature', 'fuel_price', 'cpi', 'unemployment', 'year', 'month', 'week_of_year', 'quarter']


# Create a chronological trend feature

## Create a Time-Trend Feature

A time-trend feature represents the order in which observations occurred.

Earlier observations receive lower values, while later observations receive higher values. This can help a model learn gradual changes over time.

In [51]:
# Confirming the first sorted

feature_df = (
    feature_df
    .sort_values(
        by=["date", "store"]
    )
    .reset_index(drop=True)
)

In [52]:
# Now creating the trend feature

date_order = {
    date: position
    for position, date in enumerate(
        sorted(feature_df["date"].unique())
    )
}

feature_df["time_index"] = (
    feature_df["date"].map(date_order)
)

feature_df[
    ["date", "store", "time_index"]
].head(10)

,date,store,time_index
0,2010-02-05,1,0
1,2010-02-05,2,0
2,2010-02-05,3,0
3,2010-02-05,4,0
4,2010-02-05,5,0
5,2010-02-05,6,0
6,2010-02-05,7,0
7,2010-02-05,8,0
8,2010-02-05,9,0
9,2010-02-05,10,0


The first date should is having 

time_index = 0

# Checking the new features

## Inspect the Engineered Features

The newly created date-based features are inspected to confirm that they contain the expected values.

In [53]:
# # Checking the new features

engineered_columns = [
    "date",
    "year",
    "month",
    "week_of_year",
    "quarter",
    "time_index"
]

feature_df[
    engineered_columns
].head(10)

,date,year,month,week_of_year,quarter,time_index
0,2010-02-05,2010,2,5,1,0
1,2010-02-05,2010,2,5,1,0
2,2010-02-05,2010,2,5,1,0
3,2010-02-05,2010,2,5,1,0
4,2010-02-05,2010,2,5,1,0
5,2010-02-05,2010,2,5,1,0
6,2010-02-05,2010,2,5,1,0
7,2010-02-05,2010,2,5,1,0
8,2010-02-05,2010,2,5,1,0
9,2010-02-05,2010,2,5,1,0


# Inspecting feature ranges

## Validate Feature Ranges

The minimum, maximum, and number of unique values are checked to confirm that the engineered features are logically valid.

In [54]:
#Inspecting feature ranges

feature_summary = feature_df[
    [
        "year",
        "month",
        "week_of_year",
        "quarter",
        "time_index"
    ]
].agg(
    ["min", "max", "nunique"]
).T

feature_summary

,min,max,nunique
year,2010,2012,3
month,1,12,12
week_of_year,1,52,52
quarter,1,4,4
time_index,0,142,143


## Checking for the Missing Values After Feature Engineering

Feature engineering should not introduce missing values into the dataset.

In [55]:
# Checking for the Missing Values After Feature Engineering

engineered_missing_values = (
    feature_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

engineered_missing_values

store           0
date            0
weekly_sales    0
holiday_flag    0
temperature     0
fuel_price      0
cpi             0
unemployment    0
year            0
month           0
week_of_year    0
quarter         0
time_index      0
dtype: int64

All values are 0

# Checking the original and new shapes

## Compare Dataset Shapes

The number of rows should remain unchanged because no observations were removed.

The number of columns increases because new date-based features were added.

In [56]:
# Comparing Dataset Shapes

shape_comparison = pd.DataFrame(
    {
        "Dataset": [
            "Cleaned dataset",
            "Feature-engineered dataset"
        ],
        "Rows": [
            df.shape[0],
            feature_df.shape[0]
        ],
        "Columns": [
            df.shape[1],
            feature_df.shape[1]
        ]
    }
)

shape_comparison

,Dataset,Rows,Columns
0,Cleaned dataset,6435,8
1,Feature-engineered dataset,6435,13


# Deciding how to handle the date column

Most scikit-learn regression models cannot directly use a datetime column. The useful components have already been extracted.

However, keeping the original date in the saved dataset is useful for chronological train/test splitting.

Therefore:

- Keep date in the feature-engineered CSV.
- Remove date from X later, after the chronological split.
- Do not use the raw date directly as a model input.

## Handling the Original Date Column

The original `date` column will remain in the feature-engineered dataset because it is required for chronological train-and-test splitting.

However, the raw date will not be passed directly into the regression models. The model will use the engineered date features such as `year`, `month`, `week_of_year`, `quarter`, and `time_index`.

# Separating the target and possible model features

This is only a preview. The final separation and chronological split will be performed in the preprocessing phase.

### Identifying the Target Variable

The machine-learning target remains `weekly_sales`.

All other relevant columns will act as potential predictor features.

In [57]:
# Identifying the Target Variable

target_column = "weekly_sales"

print("Target variable:", target_column)

Target variable: weekly_sales


In [58]:
# Previewing the potential feature columns

potential_feature_columns = [
    column
    for column in feature_df.columns
    if column not in [
        "weekly_sales",
        "date"
    ]
]

potential_feature_columns

['store',
 'holiday_flag',
 'temperature',
 'fuel_price',
 'cpi',
 'unemployment',
 'year',
 'month',
 'week_of_year',
 'quarter',
 'time_index']

# Classifying the feature types

### Feature-Type Identification

The features are separated conceptually according to how they should be handled during preprocessing.

- `store` is an identification category and should be treated as categorical.
- `holiday_flag` is a binary feature.
- Weather, economic, and engineered time variables are numerical.
- `weekly_sales` is the target variable.

In [59]:
# Classifying the feature types

categorical_features = [
    "store"
]

binary_features = [
    "holiday_flag"
]

numerical_features = [
    "temperature",
    "fuel_price",
    "cpi",
    "unemployment",
    "year",
    "month",
    "week_of_year",
    "quarter",
    "time_index"
]

print("Categorical features:")
print(categorical_features)

print("\nBinary features:")
print(binary_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['store']

Binary features:
['holiday_flag']

Numerical features:
['temperature', 'fuel_price', 'cpi', 'unemployment', 'year', 'month', 'week_of_year', 'quarter', 'time_index']


# NOTE: 

Although store contains numbers such as 1, 2, and 3, those numbers are store labels, not measurable quantities.

Store 20 is not mathematically “twice” Store 10. Therefore, store should be treated as a categorical feature.

## Checking for Duplicate Records

The feature-engineering process should not create duplicate store-and-date observations.

In [60]:
# Checking for Duplicated Records

duplicate_store_dates = (
    feature_df
    .duplicated(
        subset=["store", "date"]
    )
    .sum()
)

print(
    "Duplicate store-and-date records:",
    duplicate_store_dates
)

Duplicate store-and-date records: 0


## Preview the Completed Feature Dataset

The final feature-engineered dataset is inspected before saving.

In [61]:
feature_df.head()

,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment,year,month,week_of_year,quarter,time_index
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,2010,2,5,1,0
1,2,2010-02-05,2136989.46,0,40.19,2.572,210.752605,8.324,2010,2,5,1,0
2,3,2010-02-05,461622.22,0,45.71,2.572,214.424881,7.368,2010,2,5,1,0
3,4,2010-02-05,2135143.87,0,43.76,2.598,126.442065,8.623,2010,2,5,1,0
4,5,2010-02-05,317173.10,0,39.70,2.572,211.653972,6.566,2010,2,5,1,0


In [62]:
feature_df.tail()

,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment,year,month,week_of_year,quarter,time_index
6430,41,2012-10-26,1316542.59,0,41.80,3.686,199.219532,6.195,2012,10,43,4,142
6431,42,2012-10-26,514756.08,0,70.50,4.301,131.193097,6.943,2012,10,43,4,142
6432,43,2012-10-26,587603.55,0,69.17,3.506,214.741539,8.839,2012,10,43,4,142
6433,44,2012-10-26,361067.07,0,46.97,3.755,131.193097,5.217,2012,10,43,4,142
6434,45,2012-10-26,760281.43,0,58.85,3.882,192.308899,8.667,2012,10,43,4,142


## Saving the Feature-Engineered Dataset

The completed dataset is saved as `walmart_featured.csv`.

This file will be used during the preprocessing, train-test splitting, and modelling phases.

In [63]:
feature_data_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

feature_df.to_csv(
    feature_data_path,
    index=False
)

print(
    "Feature-engineered dataset saved successfully."
)

print(
    "Saved location:",
    feature_data_path.resolve()
)

Feature-engineered dataset saved successfully.
Saved location: C:\Users\Udoch\walmart-sales-regression\data\processed\walmart_featured.csv


In [64]:
# Confirming that the file exists

print(
    "File exists:",
    feature_data_path.exists()
)

print(
    "File size:",
    feature_data_path.stat().st_size,
    "bytes"
)

File exists: True
File size: 462471 bytes


## Reloading and validating the saved file

### Validate the Saved Dataset

The newly saved file is reloaded to confirm that it can be accessed successfully and contains the expected structure.

In [65]:
# Reloading and validating the saved file

saved_feature_df = pd.read_csv(
    feature_data_path
)

print(
    "Reloaded shape:",
    saved_feature_df.shape
)

saved_feature_df.head()

Reloaded shape: (6435, 13)


,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment,year,month,week_of_year,quarter,time_index
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,2010,2,5,1,0
1,2,2010-02-05,2136989.46,0,40.19,2.572,210.752605,8.324,2010,2,5,1,0
2,3,2010-02-05,461622.22,0,45.71,2.572,214.424881,7.368,2010,2,5,1,0
3,4,2010-02-05,2135143.87,0,43.76,2.598,126.442065,8.623,2010,2,5,1,0
4,5,2010-02-05,317173.10,0,39.70,2.572,211.653972,6.566,2010,2,5,1,0


# Feature Engineering Conclusion

The feature-engineering phase was completed successfully.

The following tasks were performed:

- Loaded the cleaned Walmart sales dataset.
- Converted the date column into datetime format.
- Created year, month, week-of-year, quarter, and time-index features.
- Checked the weekday feature and removed it because all records occurred on the same weekday.
- Preserved the original date column for chronological train-test splitting.
- Identified categorical, binary, and numerical feature groups.
- Confirmed that no missing values or duplicate store-date records were introduced.
- Saved the completed dataset as `data/processed/walmart_featured.csv`.

The dataset is now ready for the preprocessing and chronological train-test splitting phase.